# RecursionLens — command guide

**English** | [简体中文](https://github.com/ZiyuJiang07/recursionlens/blob/main/docs/RecursionLens_Tutorial.zh-CN.ipynb)

Every public command on one page: the decorator, the report object, its three views, its metrics, memoization, comparison, export, and error handling.

```bash
pip install recursionlens
```

> Outputs below are already saved. If Jupyter hides SVG in a downloaded notebook, click **Trust** or run all cells once.

In [3]:
import recursionlens
from recursionlens import trace

recursionlens.__version__

'0.2.0'

## 1. `@trace` — instrument a function without changing it

The decorated function is still callable, still returns the same value, and still raises the same exceptions.

In [4]:
@trace
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

fib(5)          # ordinary call, ordinary return value

5

The last call is also kept on the function as `last_report`.

In [5]:
fib.last_report.total_calls, fib.last_report.value

(15, 5)

## 2. `fn.run(*args)` — get a `TraceReport`

`run` returns a report instead of the value: default `memoize=False`, `max_calls=1000`.
Displaying the report renders the collapsible call tree.

In [6]:
report = fib.run(5)
report

TraceReport(function_name='fib', value=5, nodes=[CallNode(call_id=0, identity='1:__main__:fib(n:builtins.int=5)', signature='fib(n=5)', depth=1, status='returned', result='5', error=None, repeated=False, children=[CallNode(call_id=1, identity='1:__main__:fib(n:builtins.int=4)', signature='fib(n=4)', depth=2, status='returned', result='3', error=None, repeated=False, children=[CallNode(call_id=2, identity='1:__main__:fib(n:builtins.int=3)', signature='fib(n=3)', depth=3, status='returned', result='2', error=None, repeated=False, children=[CallNode(call_id=3, identity='1:__main__:fib(n:builtins.int=2)', signature='fib(n=2)', depth=4, status='returned', result='1', error=None, repeated=False, children=[CallNode(call_id=4, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=5, status='returned', result='1', error=None, repeated=False, children=[]), CallNode(call_id=5, identity='1:__main__:fib(n:builtins.int=0)', signature='fib(n=0)', depth=5, status='returned', result='0', error=None, repeated=False, children=[])]), CallNode(call_id=6, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=4, status='returned', result='1', error=None, repeated=True, children=[])]), CallNode(call_id=7, identity='1:__main__:fib(n:builtins.int=2)', signature='fib(n=2)', depth=3, status='returned', result='1', error=None, repeated=True, children=[CallNode(call_id=8, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=4, status='returned', result='1', error=None, repeated=True, children=[]), CallNode(call_id=9, identity='1:__main__:fib(n:builtins.int=0)', signature='fib(n=0)', depth=4, status='returned', result='0', error=None, repeated=True, children=[])])]), CallNode(call_id=10, identity='1:__main__:fib(n:builtins.int=3)', signature='fib(n=3)', depth=2, status='returned', result='2', error=None, repeated=True, children=[CallNode(call_id=11, identity='1:__main__:fib(n:builtins.int=2)', signature='fib(n=2)', depth=3, status='returned', result='1', error=None, repeated=True, children=[CallNode(call_id=12, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=4, status='returned', result='1', error=None, repeated=True, children=[]), CallNode(call_id=13, identity='1:__main__:fib(n:builtins.int=0)', signature='fib(n=0)', depth=4, status='returned', result='0', error=None, repeated=True, children=[])]), CallNode(call_id=14, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=3, status='returned', result='1', error=None, repeated=True, children=[])])]), CallNode(call_id=1, identity='1:__main__:fib(n:builtins.int=4)', signature='fib(n=4)', depth=2, status='returned', result='3', error=None, repeated=False, children=[CallNode(call_id=2, identity='1:__main__:fib(n:builtins.int=3)', signature='fib(n=3)', depth=3, status='returned', result='2', error=None, repeated=False, children=[CallNode(call_id=3, identity='1:__main__:fib(n:builtins.int=2)', signature='fib(n=2)', depth=4, status='returned', result='1', error=None, repeated=False, children=[CallNode(call_id=4, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=5, status='returned', result='1', error=None, repeated=False, children=[]), CallNode(call_id=5, identity='1:__main__:fib(n:builtins.int=0)', signature='fib(n=0)', depth=5, status='returned', result='0', error=None, repeated=False, children=[])]), CallNode(call_id=6, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=4, status='returned', result='1', error=None, repeated=True, children=[])]), CallNode(call_id=7, identity='1:__main__:fib(n:builtins.int=2)', signature='fib(n=2)', depth=3, status='returned', result='1', error=None, repeated=True, children=[CallNode(call_id=8, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=4, status='returned', result='1', error=None, repeated=True, children=[]), CallNode(call_id=9, identity='1:__main__:fib(n:builtins.int=0)', signature='fib(n=0)', depth=4, s

## 3. `report.show(mode=...)` — three views

`"tree"` is the default collapsible tree above, `"expanded"` draws every call as one SVG tree, `"dag"` merges equal arguments into subproblems.

In [7]:
report.show(mode="expanded")

In [8]:
report.show(mode="dag")

## 4. Fields, properties, and `metrics()`

Counters live on the report as attributes and properties. `metrics()` returns them as one dictionary.

In [9]:
report.metrics()

{'total_calls': 15,
 'recorded_calls': 15,
 'executed_calls': 15,
 'cached_calls': 0,
 'unique_subproblems': 6,
 'repeated_calls': 9,
 'max_depth': 5,
 'memoize': False,
 'truncated': False,
 'succeeded': True}

In [10]:
print(f"value:    {report.value}")
print(f"result:   {report.result}")
print(f"dag:      {len(report.dag_nodes)} nodes, {len(report.dag_edges)} edges")
print(f"warnings: {report.warnings}")
print(f"node 0:   {report.nodes[0].signature} -> {report.nodes[0].status}")

value:    5
result:   5
dag:      6 nodes, 8 edges
warnings: []
node 0:   fib(n=5) -> returned


## 5. `fn.with_options(...)` — memoization and recording limits

`memoize=True` caches by argument value and type; `max_calls` caps how many calls are recorded, not executed.

In [11]:
memoized = fib.with_options(memoize=True).run(5)
memoized.metrics()

{'total_calls': 9,
 'recorded_calls': 9,
 'executed_calls': 6,
 'cached_calls': 3,
 'unique_subproblems': 6,
 'repeated_calls': 3,
 'max_depth': 5,
 'memoize': True,
 'truncated': False,
 'succeeded': True}

In [12]:
capped = fib.with_options(max_calls=6).run(5)
capped.total_calls, capped.recorded_calls, capped.truncated

(15, 6, True)

## 6. `report.compare(other, labels=...)` — `ComparisonReport`

Compare two runs of the same function. The comparison also renders as HTML, with two DAGs side by side.

In [13]:
comparison = report.compare(memoized)
comparison

Metric,naive,memoized,Change
Observed calls,15,9,-6
Executed bodies,15,6,-9
Cache hits,0,3,+3
Unique subproblems,6,6,+0
Repeated calls,9,3,-6
Maximum depth,5,5,+0


In [14]:
comparison.same_result, comparison.to_dict()["delta"]

(True,
 {'total_calls': -6,
  'executed_calls': -9,
  'cached_calls': 3,
  'unique_subproblems': 0,
  'repeated_calls': -6,
  'max_depth': 0})

## 7. Export — `to_html`, `export_html`, `to_dict`, `export_json`

`to_html(mode=...)` returns a string; `export_html` writes a standalone file; `to_dict()` and `export_json()` carry the same data as JSON.

In [15]:
import tempfile
from pathlib import Path

out = Path(tempfile.mkdtemp(prefix="recursionlens-"))
paths = [
    report.export_html(out / "tree.html"),
    report.export_html(out / "dag.html", mode="dag"),
    comparison.export_html(out / "comparison.html"),
    report.export_json(out / "trace.json"),
]
len(report.to_html(mode="expanded")), [(p.name, p.stat().st_size) for p in paths]

(13401,
 [('tree.html', 5748),
  ('dag.html', 6932),
  ('comparison.html', 11867),
  ('trace.json', 8571)])

In [16]:
sorted(report.to_dict())

['dag',
 'error',
 'function_name',
 'metrics',
 'result',
 'roots',
 'schema_version',
 'warnings']

## 8. Errors — `succeeded`, `error`, and re-raise

A failing run still produces a report; the original exception is re-raised so program behavior does not change.

In [17]:
@trace
def fail(n):
    if n == 0:
        raise ValueError("boom")
    return fail(n - 1)

try:
    fail.run(3)
except ValueError as exc:
    print("re-raised:", exc)

fail.last_report.succeeded, fail.last_report.error

re-raised: boom


(False, 'ValueError: boom')

## 9. `tutor()` and `record_tutor()` — step through line by line

`tutor()` opens an interactive player: editable code, an input box for the call, and step controls. It needs `ipywidgets` (`pip install "recursionlens[tutor]"`).

```python
from recursionlens import tutor
tutor()
```

The saved output below comes from `record_tutor`, which is plain Python. The player itself is an interactive widget, so it only appears when this cell runs in a live kernel: install `ipywidgets`, run `tutor()`, then use the ⏮ ◀ ▶ ⏭ buttons or the slider to move through the recorded steps.

The recording itself is plain Python, so the same steps are available without the widget:


In [18]:
from recursionlens import record_tutor

trace = record_tutor(
    "def fib(n):\n"
    "    if n < 2:\n"
    "        return n\n"
    "    return fib(n - 1) + fib(n - 2)\n",
    "fib(4)",
)
print(trace.describe())
[(step.index, step.event, step.line, step.depth) for step in trace.steps[:6]]


call: fib(4)
steps: 36
max depth: 4
result: 3


[(0, 'call', 1, 1),
 (1, 'line', 2, 1),
 (2, 'line', 4, 1),
 (3, 'call', 1, 2),
 (4, 'line', 2, 2),
 (5, 'line', 4, 2)]

## API at a glance

| Command | Returns | Notes |
| --- | --- | --- |
| `@trace` | wrapped function | keeps return value and exceptions |
| `fn(*args)` | original value | no report from a plain call |
| `fn.run(*args)` | `TraceReport` | `memoize=False`, `max_calls=1000` |
| `fn.with_options(memoize=, max_calls=)` | `RunConfiguration` | chain `.run(*args)` |
| `fn.last_report` | `TraceReport` or `None` | updated by every call |
| `report.show(mode=)` | `None` | displays `"tree"`, `"expanded"`, `"dag"` |
| `report.to_html(mode=)` | `str` | inline HTML/SVG |
| `report.export_html(path, mode=)` | `Path` | standalone document |
| `report.metrics()` | `dict` | counters and flags |
| `report.to_dict()` / `export_json(path)` | `dict` / `Path` | schema version 1 |
| `report.compare(other, labels=)` | `ComparisonReport` | `.same_result`, `.show()`, `.to_dict()`, `.export_html()` |
| `report.value` / `.result` / `.error` | Python value / `str` / `str` | top-level outcome |
| `report.total_calls`, `.recorded_calls`, `.executed_calls`, `.cached_calls` | `int` | call counters |
| `report.unique_subproblems`, `.repeated_calls`, `.max_depth` | `int` | DAG and depth metrics |
| `report.dag_nodes` / `.dag_edges` | `list` | merged subproblems |
| `report.warnings`, `.truncated`, `.succeeded`, `.memoize` | `list` / `bool` | run flags |
| `tutor(code=, call=)` | `TutorPlayer` widget | needs `ipywidgets`; step controls and live stack |
| `record_tutor(code, call)` | `TutorTrace` | `.steps`, `.describe()`, `.error`; no widget needed |

Two limits worth knowing: memoization needs hashable arguments (otherwise a warning is recorded and those calls are not cached), and RecursionLens executes your function — it is not a sandbox.